# 32. spaCy Language Pipelines — `spacy`
**Python Libraries Lab · Phase 2 · Natural Language Processing**

_Fast, production-ready text processing: entities, grammar and rules._

⚙️ **Runs in Colab** after the setup cell below.

spaCy is built for real applications. You load a trained pipeline once, then pass it text, and you get back a document where every word already knows its lemma, its part of speech, how it connects to other words, and whether it is part of a named entity like a person, a city or an amount of money. It is fast enough to process millions of reviews.

**In this notebook:** You will analyse ShopEasy support messages, extract people, places, products and amounts, pull out noun phrases that show what customers talk about, write a rule that finds order numbers and money amounts, and draw the entities on the text.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
spaCy needs a trained English model. The next cell downloads the small one (about 13 MB).

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q spacy
import importlib.metadata as md
for pkg in ['spacy']:
    print(pkg, md.version(pkg))

In [ ]:
!python -m spacy download en_core_web_sm -q

## Step 1 · Load the pipeline and inspect tokens
Each token knows its lemma, part of speech and grammatical role.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("Pipeline:", nlp.pipe_names)

doc = nlp("Priya ordered two yoga mats from ShopEasy in Pune for Rs. 2,598 last Friday.")
for t in doc:
    print(f"{t.text:<9} lemma={t.lemma_:<8} pos={t.pos_:<6} dep={t.dep_}")

## Step 2 · Named entities
People, places, organisations, money, dates...

In [ ]:
messages = [
    "Hi, I'm Rahul Verma from Delhi. My Sony headphones cost Rs. 4,999 and stopped working on 3 March.",
    "Asha in Mumbai wants a refund of 1,299 rupees for order SE-2026-0042.",
    "Can Amazon or Flipkart deliver faster than ShopEasy to Bengaluru by Monday?",
]
for doc in nlp.pipe(messages):
    print([(e.text, e.label_) for e in doc.ents])

## Step 3 · What are people talking about?
Noun chunks are short noun phrases like 'my Sony headphones'.

In [ ]:
from collections import Counter
chunks = Counter(c.root.lemma_.lower() for doc in nlp.pipe(messages) for c in doc.noun_chunks)
print(chunks.most_common(6))

## Step 4 · Rules with the Matcher
Find order numbers and rupee amounts with token patterns.

In [ ]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)
matcher.add("ORDER_ID", [[{"TEXT": {"REGEX": r"^SE-\d{4}-\d{4}$"}}]])
matcher.add("RUPEES", [[{"LOWER": {"IN": ["rs", "rs."]}}, {"IS_PUNCT": True, "OP": "?"}, {"LIKE_NUM": True}],
                       [{"LIKE_NUM": True}, {"LOWER": {"IN": ["rupees", "inr"]}}]])
for doc in nlp.pipe(messages):
    found = [(nlp.vocab.strings[mid], doc[s:e].text) for mid, s, e in matcher(doc)]
    print(found)

## Step 5 · See it with displaCy
Colab shows the entities highlighted inside the text.

In [ ]:
from spacy import displacy
displacy.render(nlp(messages[0]), style="ent", jupyter=True)

## ✍️ Your turn
Store a **list of the entity labels** (like `['PERSON', 'GPE']`) that spaCy finds in `'Meera from Chennai paid 500 rupees.'` in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    exp = [e.label_ for e in nlp("Meera from Chennai paid 500 rupees.").ents]
    return answer == exp, f"Expected {exp}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = [e.label_ for e in nlp("Meera from Chennai paid 500 rupees.").ents]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **nlp pipeline**
- **Doc / Token**
- **Lemma**
- **Named entity**
- **Matcher**
- **displaCy**

**Next:** 33. Gensim Topics & Word Vectors (`gensim`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/33_gensim.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install spacy
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.